In [2]:
import os
import joblib
import requests
import random
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# 1. Load Groq API Key from RAG-part/.env
load_dotenv("RAG-part/.env")

# 2. Load the trained Intent Classifier from root
intent_model = joblib.load("intent_classifier/intent_classification_hybrid_model.joblib")

# 3. Initialize Groq LLM
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0.0
)

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize embedding model (384 dimensions)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Load FAISS index from RAG-part folder
vector_db = FAISS.load_local(
    folder_path="RAG-part/vector-db",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

C:\Users\Omkar Raut\AppData\Local\Temp\ipykernel_14752\3124338010.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
C:\Users\Omkar Raut\AppData\Local\Temp\ipykernel_14752\3124338010.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\Omkar Raut\OneDrive\Desktop\SIH-OIL\oil-sif\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and i

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# A. Rephrase Chain for Contextual Retrieval
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and the latest user question, formulate a standalone question. Do NOT answer it, just rewrite if needed."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])
rephrase_chain = contextualize_prompt | llm

# B. Professional Customer Email System Prompt
email_system_prompt = """You are an official Airline Customer Support Representative for IndiGo.

Write your response as a formal, courteous, and helpful customer support email.
Follow this email structure strictly:
1. Salutation: "Dear Passenger," (or use their name if provided).
2. Opening: Briefly thank them for contacting customer support.
3. Body: Clearly explain the policy details strictly using the provided Context. Use bullet points for limits, fees, or steps.
4. Closing: "If you have any further questions or require additional assistance, please feel free to reply to this email."
5. Sign-off:
   Warm regards,
   IndiGo Customer Support Team
   24/7 Helpline: 0124-6173838 | www.goindigo.in

Strict Grounding Rules:
- Only include facts directly supported by the Context.
- Never invent fees, baggage weights, or refund policies.
- If the Context lacks the answer, politely write:
  "We are currently reviewing your specific request with our ticketing department. A customer support representative will follow up with you shortly."

Context:
{context}
"""

email_prompt_template = ChatPromptTemplate.from_messages([
    ("system", email_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

chat_history = []

def ask_airline_assistant_with_history(question, k=4):
    if not chat_history:
        search_query = question
    else:
        search_query = rephrase_chain.invoke({
            "chat_history": chat_history,
            "question": question
        }).content

    documents = vector_db.similarity_search(search_query, k=k)
    context_text = "\n\n---\n\n".join(doc.page_content for doc in documents)

    format_prompt = email_prompt_template.invoke({
        "context": context_text,
        "chat_history": chat_history,
        "question": question
    })

    response = llm.invoke(format_prompt)
    sources = list(set(doc.metadata.get("source", "Unknown") for doc in documents))

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))

    return {
        "answer": response.content,
        "sources": sources
    }

In [7]:
MOCK_FLIGHTS = {
    "6E-204": {"route": "Delhi (DEL) to Mumbai (BOM)", "time": "14:30", "status": "ON TIME", "gate": "14", "terminal": "T2"},
    "6E-551": {"route": "Bengaluru (BLR) to Goa (GOI)", "time": "18:00", "status": "DELAYED (New departure: 18:45)", "gate": "06", "terminal": "T1"},
    "6E-101": {"route": "Mumbai (BOM) to Dubai (DXB)", "time": "21:15", "status": "ON TIME", "gate": "08", "terminal": "T2"}
}

MOCK_BOOKINGS = {
    "IND123": {"name": "Rahul Sharma", "flight": "6E-204", "route": "DEL -> BOM", "seat": "12A", "status": "CONFIRMED"},
    "FLY789": {"name": "Priya Patel", "flight": "6E-551", "route": "BLR -> GOI", "seat": "07C", "status": "CHECKED-IN"}
}

MOCK_PRICES = {
    "DEL-BOM": "₹4,499 (Standard Economy) / ₹6,299 (Flexi Plus)",
    "BOM-GOI": "₹3,150 (Standard Economy) / ₹4,800 (Flexi Plus)",
    "BLR-DEL": "₹5,200 (Standard Economy) / ₹7,100 (Flexi Plus)"
}

def mock_airline_service(intent, user_message):
    msg_upper = user_message.upper()
    
    if "status" in intent or "schedule" in intent:
        for flight_no, details in MOCK_FLIGHTS.items():
            if flight_no in msg_upper or flight_no.replace("-", "") in msg_upper:
                return (
                    f"Dear Passenger,\n\n"
                    f"Thank you for contacting IndiGo regarding flight {flight_no}.\n\n"
                    f"Here are your latest flight details:\n"
                    f"• Flight Number: {flight_no}\n"
                    f"• Route: {details['route']}\n"
                    f"• Scheduled Departure: {details['time']}\n"
                    f"• Current Status: {details['status']}\n"
                    f"• Terminal & Gate: Terminal {details['terminal']}, Gate {details['gate']}\n\n"
                    f"Please arrive at the boarding gate at least 25 minutes prior to departure.\n\n"
                    f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
                )
        return (
            "Dear Passenger,\n\n"
            "Thank you for contacting IndiGo flight status service.\n\n"
            "Our scheduled domestic flights are currently operating on schedule. "
            "To view the real-time status of your specific flight, please reply with your "
            "flight number (e.g., 6E-204) or visit www.goindigo.in/flight-status.\n\n"
            "Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )
    
    elif "reservation" in intent or "details" in intent:
        for pnr, b in MOCK_BOOKINGS.items():
            if pnr in msg_upper:
                return (
                    f"Dear {b['name']},\n\n"
                    f"Thank you for contacting IndiGo reservations.\n\n"
                    f"Here are your confirmed reservation details (PNR: {pnr}):\n"
                    f"• Passenger Name: {b['name']}\n"
                    f"• Flight: {b['flight']} ({b['route']})\n"
                    f"• Seat: {b['seat']}\n"
                    f"• Status: {b['status']}\n\n"
                    f"Web check-in opens 48 hours prior to departure.\n\n"
                    f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
                )
        return (
            "Dear Passenger,\n\n"
            "We could not locate an active booking with the reference provided. "
            "Please verify your 6-character PNR (e.g., IND123) and reply to this email.\n\n"
            "Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )
        
    elif "prices" in intent:
        return (
            f"Dear Passenger,\n\n"
            f"Thank you for inquiring about IndiGo flight fares.\n\n"
            f"Current starting fares on popular routes:\n"
            f"• Delhi to Mumbai: {MOCK_PRICES['DEL-BOM']}\n"
            f"• Mumbai to Goa: {MOCK_PRICES['BOM-GOI']}\n"
            f"• Bengaluru to Delhi: {MOCK_PRICES['BLR-DEL']}\n\n"
            f"Fares include standard 7 kg cabin baggage and 15 kg checked baggage.\n\n"
            f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )
        
    else:
        new_pnr = "6E" + str(random.randint(1000, 9999))
        return (
            f"Dear Passenger,\n\n"
            f"Thank you for initiating your flight booking with IndiGo.\n\n"
            f"Your provisional reservation has been generated:\n"
            f"• Provisional PNR: {new_pnr}\n"
            f"• Booking Status: Awaiting Payment\n\n"
            f"Please complete your reservation within 2 hours at www.goindigo.in/pay.\n\n"
            f"Warm regards,\nIndiGo Customer Support Team\nwww.goindigo.in"
        )

def generate_email_subject(intent, resolved=True):
    clean_topic = intent.replace("check_", "").replace("get_", "").replace("_", " ").title()
    if resolved:
        return f"IndiGo Customer Support: Your Inquiry Regarding {clean_topic}"
    else:
        return f"[ESCALATION REQUIRED] Support Ticket: {clean_topic}"

In [8]:
N8N_PRODUCTION_URL = os.environ["PRODUCTION_URL"]

def trigger_n8n_webhook(customer_email, subject, answer, resolved):
    payload = {
        "customer_email": customer_email,
        "subject": subject,
        "answer": answer,
        "resolved": resolved
    }
    try:
        response = requests.post(N8N_PRODUCTION_URL, json=payload, timeout=10)
        print(f"[n8n Webhook Response]: HTTP {response.status_code}")
        return response.status_code == 200
    except Exception as e:
        print(f"[n8n Webhook Error]: {e}")
        return False

In [9]:
TRANSACTIONAL_INTENTS = {
    "check_flight_status",
    "check_flight_schedule",
    "check_flight_reservation",
    "check_trip_details",
    "check_flight_prices",
    "check_trip_prices",
    "book_flight",
    "book_trip"
}

def process_customer_ticket(customer_email, user_message):
    print("\n" + "="*60)
    print(f"INCOMING TICKET FROM: {customer_email}")
    print(f"MESSAGE: \"{user_message}\"")
    print("="*60)
    
    # 1. Predict Intent & Confidence
    intent = intent_model.predict([user_message])[0]
    confidence = max(intent_model.predict_proba([user_message])[0])
    print(f"[ML Classifier]: Intent = '{intent}' | Confidence = {confidence:.2%}")
    
    # 2. Triage Logic
    # Case A: Low Confidence or Human Agent Request
    if confidence < 0.50 or intent == "human_agent":
        resolved = False
        subject = generate_email_subject(intent, resolved=False)
        answer = (
            f"*** INTERNAL ESCALATION TICKET ***\n\n"
            f"Customer Email: {customer_email}\n"
            f"Detected Intent: {intent} (Confidence: {confidence:.1%})\n"
            f"Customer Message: \"{user_message}\"\n\n"
            f"Reason for Escalation:\n"
            f"• Query requires human intervention or confidence fell below 0.50.\n\n"
            f"Action Required:\n"
            f"Please review the customer's request and follow up directly."
        )
        print("[Routing]: Low confidence / Human request -> Escalated to rautom41105@gmail.com")
        
    # Case B: Transactional Intent -> Mock API
    elif intent in TRANSACTIONAL_INTENTS:
        resolved = True
        subject = generate_email_subject(intent, resolved=True)
        answer = mock_airline_service(intent, user_message)
        print("[Routing]: Transactional intent -> Handled by Mock Airline API")
        
    # Case C: Policy Intent -> RAG Knowledge Base
    else:
        print("[Routing]: Policy intent -> Querying RAG Engine...")
        rag_output = ask_airline_assistant_with_history(user_message, k=4)
        answer = rag_output["answer"]
        
        if "I do not have specific policy information" in answer or "reviewing your specific request" in answer:
            resolved = False
            subject = generate_email_subject("unresolved_policy_query", resolved=False)
            answer = (
                f"*** INTERNAL ESCALATION TICKET - POLICY GAP ***\n\n"
                f"Customer Email: {customer_email}\n"
                f"Customer Message: \"{user_message}\"\n\n"
                f"Reason for Escalation:\n"
                f"• Policy information is missing from the RAG knowledge base.\n\n"
                f"Action Required:\n"
                f"Please assist the customer directly with verified airline guidelines."
            )
            print("[Routing]: Policy not found -> Escalated to rautom41105@gmail.com")
        else:
            resolved = True
            subject = generate_email_subject(intent, resolved=True)
            print("[Routing]: Policy resolved -> Sending email to customer")
            
    # 3. Dispatch to n8n Webhook
    dispatch_success = trigger_n8n_webhook(customer_email, subject, answer, resolved)
    
    return {
        "customer_email": customer_email,
        "intent": intent,
        "confidence": confidence,
        "resolved": resolved,
        "subject": subject,
        "answer": answer,
        "dispatched": dispatch_success
    }

print("Master Orchestrator ready!")

Master Orchestrator ready!


In [17]:
test1 = process_customer_ticket(
    customer_email="rautom41105@gmail.com",
    user_message="Can I get a refund for my cancelled flight?"
)


INCOMING TICKET FROM: rautom41105@gmail.com
MESSAGE: "Can I get a refund for my cancelled flight?"
[ML Classifier]: Intent = 'get_refund' | Confidence = 97.45%
[Routing]: Policy intent -> Querying RAG Engine...
[Routing]: Policy resolved -> Sending email to customer
[n8n Webhook Response]: HTTP 200


In [12]:
test2 = process_customer_ticket(
    customer_email="rautom41105@gmail.com",
    user_message="Can you please check the status of flight 6E-204?"
)


INCOMING TICKET FROM: rautom41105@gmail.com
MESSAGE: "Can you please check the status of flight 6E-204?"
[ML Classifier]: Intent = 'check_flight_status' | Confidence = 96.90%
[Routing]: Transactional intent -> Handled by Mock Airline API
[n8n Webhook Response]: HTTP 200


In [13]:
test3 = process_customer_ticket(
    customer_email="rautom41105@gmail.com",
    user_message="I have an urgent issue with my booking and I need to speak to a human representative right now!"
)


INCOMING TICKET FROM: rautom41105@gmail.com
MESSAGE: "I have an urgent issue with my booking and I need to speak to a human representative right now!"
[ML Classifier]: Intent = 'human_agent' | Confidence = 96.75%
[Routing]: Low confidence / Human request -> Escalated to rautom41105@gmail.com
[n8n Webhook Response]: HTTP 200
